# 3.9: The recipe

With **BF16**, the range matches FP32's, so gradient underflow is rarely a problem and loss scaling is usually unnecessary. Kalamkar et al. (2019) studied BF16 training across image classification, speech recognition, language modeling, and other workloads, and emphasized that because BF16 keeps FP32's range, no hyperparameter tuning is required for convergence, unlike FP16. This simplicity is why BF16 is the common choice for training large models on hardware that supports it. In PyTorch, `torch.autocast` handles the casting (keeping sensitive operations in FP32 automatically), and for FP16 a `torch.amp.GradScaler` handles dynamic loss scaling:

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.

This snippet uses `loss_fn`, `x`, `y` from the surrounding training code. Those names are not defined in an earlier listing, so the cell shows the pattern as it appears in the chapter.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From Warmup plus decay for LLM pretraining
import math

def lr_at(step, peak_lr, warmup_steps, total_steps, min_lr_ratio=0.1):
    min_lr = peak_lr * min_lr_ratio
    if step < warmup_steps:
        return peak_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    progress = min(progress, 1.0)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))

# PyTorch: apply via LambdaLR, which multiplies the optimizer's base lr
import torch

model = torch.nn.Linear(10, 10)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda step: lr_at(step, 1.0, warmup_steps=1000, total_steps=100_000))

# From A baseline recipe
import math, torch, torch.nn as nn, torch.nn.functional as F


In [ ]:
scaler = torch.amp.GradScaler("cuda")         # only needed for FP16
with torch.autocast(device_type="cuda", dtype=torch.float16):
    loss = loss_fn(model(x), y)
scaler.scale(loss).backward()
scaler.unscale_(opt)                           # so clipping sees true gradients
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(opt)                               # skips the step if grads overflowed
scaler.update()
